## 데이터 수집 자동화

### 안정적 수집과 저장

#### 수집 오류 유형

- 요소 없음 : CSS선택자가 틀렸거나 화면에 없음 > 예외처리
- 로딩 지연 : 페이지가 늦게 뜸 > 명시적 대기
- 클릭 실패 : 버튼이 아직 클릭 불가 > 명시적 대기
- 데이터 누락 : 일부 항목이 페이지에 없음 > 기본값 처리
- 중복 수집 : 같은 데이터가 포함 > 판다스가 중복처리
- 저장 실패 : 파일 존재여부, 권한여부 > 파일명 변경, 예외처리

In [17]:
import pandas as pd
import os

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [12]:
try:         # 실행중 오류가 날 것 같은 소스코드
    price = input('가격입력 > ')
    print(int(price))
except Exception as e:    # 실행중 오류가 발생하면 처리할 부분
    print(f'가격은 숫자만 입력할 수 있습니다.')
finally:        # 실행중 오류가 나든 안나든 무조건 실행하는 부분 
    print('프로그램 종료')    

323231234
프로그램 종료


In [13]:
from selenium import webdriver

In [15]:
driver = webdriver.Chrome()

try:
    driver.get('https://www.google.com') # 웹사이트 오픈
    print(driver.title)  # 웹사이트의 head 영역에 있는 탭이름(title) 가져오기
except Exception as e:
    print(f'실행중 오류발생! : {e.args}')
finally:
    driver.quit()        

Google


#### find_element 문제점
- find_elements() : 실행중 오류 발생 X
- find_element() : 실행중 오류 발생 가능서 높음
    - 함수를 만드는게 유리

In [ ]:
# 안전하게 요소 추출함수
def get_text_safe(parent, selector, default=''):
    try:
        return parent.find_element(By.CSS_SELECTOR, selector).text
    except:
        return default # 내가 지정한 기본값이 그냥 리턴

In [21]:
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 10)
driver.get('https://www.danawa.com')

wait.until(
    EC. visibility_of_element_located((By.ID, 'danawa_footer'))
)

<selenium.webdriver.remote.webelement.WebElement (session="4590ce3a79c5990c47976ccdf6489612", element="f.3BD4B7CEBEA08897DE87EDB7480AEA27.d.CE8661C41DD52EA091B50D85EDFCEF4F.e.73")>

In [26]:
get_text_safe(driver, 'div.item > a .txt', 'None')

'최근'

#### 명시적 대기 타임 아웃

- 명시적 대기를 하더라도 시간을 초과하면 실행 중 오류가 발생
- 실행 중 오류 잡으려면 try ~ except 사용 필요

In [32]:
from selenium.common.exceptions import TimeoutException

try:
    driver.get('https://www.naver.com/')
    wait.until(
        # 네이버는 footer가 아아디. #footer
        EC.visibility_of_element_located((By.CSS_SELECTOR, '.footer'))
    )
    print('로딩 완료!')
except TimeoutException as e:
    print('로딩 실패! 타임오버')

로딩 실패! 타임오버


#### 버튼 클릭 가능상태까지 대기

- 버튼, 링크가 화면에 나타나도 js가 로딩완료될때까지 클릭못하는 경우가 발생할 수 있음
- 클릭상태까지 대기

In [33]:
enable_button = wait.until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, '#account a'))
)

enable_button.click()

#### 로그파일 저장

- 화면에 출력보다 로그파일 저장이 효율적

In [41]:
import logging

logging.basicConfig(
    filename = 'scraping.log',
    level = logging.ERROR,      # DEBUG, INFO, WARN, WARNING, ERROR, FATAL...
    encoding = 'utf-8'
)

In [42]:
logging.info('수집시작')
logging.info('상품 데이터 수집완료')
logging.error('수집 중 오류발생!!')

--- Logging error ---
Traceback (most recent call last):
  File "c:\Program Files\Python314\Lib\logging\__init__.py", line 1151, in emit
    msg = self.format(record)
  File "c:\Program Files\Python314\Lib\logging\__init__.py", line 999, in format
    return fmt.format(record)
           ~~~~~~~~~~^^^^^^^^
  File "c:\Program Files\Python314\Lib\logging\__init__.py", line 715, in format
    s = self.formatMessage(record)
  File "c:\Program Files\Python314\Lib\logging\__init__.py", line 684, in formatMessage
    return self._style.format(record)
           ~~~~~~~~~~~~~~~~~~^^^^^^^^
  File "c:\Program Files\Python314\Lib\logging\__init__.py", line 472, in format
    return self._format(record)
           ~~~~~~~~~~~~^^^^^^^^
  File "c:\Program Files\Python314\Lib\logging\__init__.py", line 468, in _format
    return self._fmt % values
           ~~~~~~~~~~^~~~~~~~
ValueError: incomplete format
Call stack:
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", 

### DB에 저장

- 이전 저장한 다나와 데이터 사용

In [ ]:
df = pd.read_csv('../data/danawa_게이밍PC.csv', encoding='utf-8')
df.drop(columns=["Unnamed:0"], errors='ignore')

,Unnamed: 0,index,computer_name,computer_spec,price_spec,price
0,0,0,영웅컴퓨터 영웅 울트라게이밍 9800XS R7 9800X3D RTX5080,9800X3D / RTX 5080 / (AMD) B650 / OS미포함 / 900W...,"32GB, M.2 1TB",4523480.0
1,1,1,영웅컴퓨터 영웅 875 게이밍울트라560X R5 5600 RTX5060,5600 / RTX 5060 / (AMD) B550 / OS미포함 / 600W / ...,"16GB, M.2 256GB",1181800.0
2,2,2,포유컴퓨터 퍼포먼스PC 47 R5 7500F RTX5060Ti,7500F / RTX 5060 Ti / (AMD) B650 / OS미포함 / 700...,"16GB, M.2 500GB",1579200.0
3,3,3,MSI MAG Codex Z2 A9NVR A18 R7-9800X3D RTX 5070...,9800X3D / RTX 5070 Ti / (AMD) B850 / OS미포함 / 8...,"32GB, M.2 1TB",3943540.0
4,4,4,영웅컴퓨터 영웅 게이밍울트라 7857 R7 7800X3D RTX5070,7800X3D / RTX 5070 / (AMD) B650 / OS미포함 / 800W...,"32GB, M.2 1TB",2805860.0
...,...,...,...,...,...,...
894,895,895,HP 오멘 35L GT16-0015KL Win11Pro,285K / RTX 5080 / (인텔) Z890 / 윈도우11 프로 / 1000W...,"64GB, M.2 2TB",7310000.0
895,896,896,HP 옴니데스크 M03-0038kr Win11Pro,225 / 인텔 그래픽스 / 윈도우11 프로 / 500W / 인텔 / 코어울트라 시...,"16GB, M.2 1TB",1550450.0
896,897,897,엔피씨코리아 NEXPC 게이밍프로 G365,9700X / RTX 5060 / (AMD) B650 / OS미포함 / 800W /...,"32GB, M.2 1TB",2205720.0
897,898,898,다나와표준PC 게임용 260615,7500X3D / RTX 5070 / (AMD) B850 / OS미포함 / 850W...,"16GB, M.2 1TB",2799990.0


In [46]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 899 entries, 0 to 898
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Unnamed: 0     899 non-null    int64  
 1   index          899 non-null    int64  
 2   computer_name  899 non-null    str    
 3   computer_spec  899 non-null    str    
 4   price_spec     899 non-null    str    
 5   price          899 non-null    float64
dtypes: float64(1), int64(2), str(3)
memory usage: 42.3 KB


In [47]:
df

,Unnamed: 0,index,computer_name,computer_spec,price_spec,price
0,0,0,영웅컴퓨터 영웅 울트라게이밍 9800XS R7 9800X3D RTX5080,9800X3D / RTX 5080 / (AMD) B650 / OS미포함 / 900W...,"32GB, M.2 1TB",4523480.0
1,1,1,영웅컴퓨터 영웅 875 게이밍울트라560X R5 5600 RTX5060,5600 / RTX 5060 / (AMD) B550 / OS미포함 / 600W / ...,"16GB, M.2 256GB",1181800.0
2,2,2,포유컴퓨터 퍼포먼스PC 47 R5 7500F RTX5060Ti,7500F / RTX 5060 Ti / (AMD) B650 / OS미포함 / 700...,"16GB, M.2 500GB",1579200.0
3,3,3,MSI MAG Codex Z2 A9NVR A18 R7-9800X3D RTX 5070...,9800X3D / RTX 5070 Ti / (AMD) B850 / OS미포함 / 8...,"32GB, M.2 1TB",3943540.0
4,4,4,영웅컴퓨터 영웅 게이밍울트라 7857 R7 7800X3D RTX5070,7800X3D / RTX 5070 / (AMD) B650 / OS미포함 / 800W...,"32GB, M.2 1TB",2805860.0
...,...,...,...,...,...,...
894,895,895,HP 오멘 35L GT16-0015KL Win11Pro,285K / RTX 5080 / (인텔) Z890 / 윈도우11 프로 / 1000W...,"64GB, M.2 2TB",7310000.0
895,896,896,HP 옴니데스크 M03-0038kr Win11Pro,225 / 인텔 그래픽스 / 윈도우11 프로 / 500W / 인텔 / 코어울트라 시...,"16GB, M.2 1TB",1550450.0
896,897,897,엔피씨코리아 NEXPC 게이밍프로 G365,9700X / RTX 5060 / (AMD) B650 / OS미포함 / 800W /...,"32GB, M.2 1TB",2205720.0
897,898,898,다나와표준PC 게임용 260615,7500X3D / RTX 5070 / (AMD) B850 / OS미포함 / 850W...,"16GB, M.2 1TB",2799990.0


#### DB 저장

- SQLAlchemy 패키지 사용. SQL 쿼리문 없이 CRUD 가능
- `pip install SQLAlchemy`

In [48]:
!pip install SQLAlchemy

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.4 MB ? eta -:--:--
   ----------------- ---------------------- 1.0/2.4 MB 3.9 MB/s eta 0:00:01
   -------------------------------------- - 2.4/2.4 MB 4.4 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 4.3 MB/s  0:00:00


In [ ]:
from datetime import datetime
from getpass import getpass

# SQLAlchemy
import psycopg
from sqlalchemy import create_engine, URL, Text, BigInteger, Integer

In [50]:
df.head()

,Unnamed: 0,index,computer_name,computer_spec,price_spec,price
0,0,0,영웅컴퓨터 영웅 울트라게이밍 9800XS R7 9800X3D RTX5080,9800X3D / RTX 5080 / (AMD) B650 / OS미포함 / 900W...,"32GB, M.2 1TB",4523480.0
1,1,1,영웅컴퓨터 영웅 875 게이밍울트라560X R5 5600 RTX5060,5600 / RTX 5060 / (AMD) B550 / OS미포함 / 600W / ...,"16GB, M.2 256GB",1181800.0
2,2,2,포유컴퓨터 퍼포먼스PC 47 R5 7500F RTX5060Ti,7500F / RTX 5060 Ti / (AMD) B650 / OS미포함 / 700...,"16GB, M.2 500GB",1579200.0
3,3,3,MSI MAG Codex Z2 A9NVR A18 R7-9800X3D RTX 5070...,9800X3D / RTX 5070 Ti / (AMD) B850 / OS미포함 / 8...,"32GB, M.2 1TB",3943540.0
4,4,4,영웅컴퓨터 영웅 게이밍울트라 7857 R7 7800X3D RTX5070,7800X3D / RTX 5070 / (AMD) B650 / OS미포함 / 800W...,"32GB, M.2 1TB",2805860.0


In [ ]:
# 저장할 테이블명 - danawa_gaming_pc_yymmdd
table_name = 'danawa_gaming_pc_' + datetime.now().strftime('%y%m%d')
print(table_name)

danawa_gaming_pc_261006


In [64]:
# PostgreSQL 연결 정보
db_url = URL.create(
    "postgresql+psycopg",
    username='postgres',
    password='123456',
    host = '127.0.0.1',
    port = 5432,
    database = 'postgres'
)

In [65]:
engine = create_engine(db_url)

In [66]:
# 테이블 생성하고 DataFrame 데이터 저장
df.to_sql(
    name = table_name,
    con = engine,
    if_exists = 'fail',
    index = False,
    dtype = {
        'index' : BigInteger(),
        'computer_name' : Text(),
        'computer_spec' : Text(),
        'price_spec' : Text(),
        'price' : Integer()        
    }
)

engine.dispose()  # 사용후 해제

print(f'{len(df)}건을 저장했습니다.')

899건을 저장했습니다.


#### 스케줄링

- 정해진 시간마다 반복 수행하도록 하는 것
- OS 마다 방식이 상이
    - Windows : 작업 스케줄러
    - Linux : cron
    - Python : shedule 라이브러리 가능